---

## 2. Data Loading Utility

We define a function to load the dataset. The script is configured to handle the **Pima Indians Diabetes** dataset for classification and the **Energy Efficiency** dataset for regression.

In [21]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error, accuracy_score, confusion_matrix
from sklearn.preprocessing import Normalizer
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, export_text
from sklearn.ensemble import (RandomForestClassifier, RandomForestRegressor,
                              AdaBoostClassifier, AdaBoostRegressor,
                              GradientBoostingClassifier, GradientBoostingRegressor)
import xgboost as xgb

# Set random seed for reproducibility
random_seed = 42

XGBoostError: 
XGBoost Library (libxgboost.so) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["/home/rohit/anaconda3/lib/python3.11/site-packages/zmq/backend/cython/../../../../.././libstdc++.so.6: version `CXXABI_1.3.15' not found (required by /home/rohit/anaconda3/lib/libxgboost.so)"]


---

## 3. The Modeling Functions

We split the models into two functions: one for standard `scikit-learn` algorithms and one for `XGBoost`.

### Scikit-Learn Ensembles

In [22]:
def read_data(run_num, prob):
    normalise = False

    if prob == 'classifification':
        # Source: Pima-Indian diabetes dataset
        # data_in = np.genfromtxt("data/pima.csv", delimiter=",")
        # For this tutorial, we use a placeholder or local path
        data_in = np.genfromtxt("data/pima.csv", delimiter=",")
        data_inputx = data_in[:, 0:8]
        data_inputy = data_in[:, -1]

    elif prob == 'regression':
        # Source: Energy efficiency dataset
        data_in = np.genfromtxt('data/ENB2012_data.csv', delimiter=",")
        data_inputx = data_in[:, 0:8]
        data_inputy = data_in[:, 8]

    if normalise:
        transformer = Normalizer().fit(data_inputx)
        data_inputx = transformer.transform(data_inputx)

    return train_test_split(data_inputx, data_inputy, test_size=0.40, random_state=run_num)

In [23]:
def scipy_models(x_train, x_test, y_train, y_test, type_model, hidden, learn_rate, run_num, problem):
    tree_depth = 2

    if problem == 'classifification':
        if type_model == 0: # Neural Network (MLP)
            model = MLPClassifier(hidden_layer_sizes=(hidden,), random_state=run_num, max_iter=100, solver='sgd', learning_rate_init=learn_rate)
        elif type_model == 1: # Decision Tree
            model = DecisionTreeClassifier(random_state=0, max_depth=tree_depth)
        elif type_model == 2: # Random Forest
            model = RandomForestClassifier(n_estimators=100, max_depth=tree_depth, random_state=run_num)
        elif type_model == 3: # AdaBoost
            model = AdaBoostClassifier(n_estimators=100, random_state=run_num)
        elif type_model == 4: # Gradient Boosting
            model = GradientBoostingClassifier(n_estimators=10, random_state=run_num)

    elif problem == 'regression':
        if type_model == 0:
            model = MLPRegressor(hidden_layer_sizes=(hidden*3,), random_state=run_num, max_iter=500, solver='adam', learning_rate_init=learn_rate)
        elif type_model == 1:
            model = DecisionTreeRegressor(random_state=0, max_depth=tree_depth)
        elif type_model == 2:
            model = RandomForestRegressor(n_estimators=100, max_depth=tree_depth, random_state=run_num)
        elif type_model == 3:
            model = AdaBoostRegressor(n_estimators=100, random_state=run_num)
        elif type_model == 4:
            model = GradientBoostingRegressor(n_estimators=10, random_state=run_num)

    model.fit(x_train, y_train)
    y_pred_test = model.predict(x_test)

    if problem == 'regression':
        return np.sqrt(mean_squared_error(y_test, y_pred_test))
    else:
        return accuracy_score(y_test, y_pred_test)

### XGBoost Implementation

In [24]:
def xgboost_models(x_train, x_test, y_train, y_test, run_num, problem):
    if problem == 'classifification':
        model = xgb.XGBClassifier(colsample_bytree=0.3, learning_rate=0.1, max_depth=5, alpha=5, n_estimators=100)
    else:
        model = xgb.XGBRegressor(objective='reg:linear', colsample_bytree=0.3, learning_rate=0.1, max_depth=5, alpha=5, n_estimators=100)

    model.fit(x_train, y_train)
    y_pred_test = model.predict(x_test)

    if problem == 'regression':
        return np.sqrt(mean_squared_error(y_test, y_pred_test))
    else:
        return accuracy_score(y_test, y_pred_test)

---

## 4. Run Experiment

This block executes the training loop and prints the statistical performance (Mean and Standard Deviation) for each model.

To use this code in a Jupyter Notebook, you can copy the blocks below into separate cells. This structure organizes the script into a logical flow for data experimentation.

---

# Tutorial: Ensemble Methods for Classification and Regression

This notebook provides a comparative analysis of various machine learning models, from simple Decision Trees to advanced Ensemble methods like Random Forest, AdaBoost, Gradient Boosting, and XGBoost.

## 1. Environment Setup

First, we import the necessary libraries for data processing, model building, and evaluation.

In [25]:
# Experiment Parameters
max_expruns = 5
prob = 'classifification' # Change to 'regression' for RMSE results
learn_rate = 0.01
hidden = 8

# Accuracy/RMSE Storage
results = { "MLP": [], "DecisionTree": [], "RandomForest": [], "AdaBoost": [], "GBoost": [], "XGBoost": [] }

print(f"Starting experiments for: {prob}\n")

for run_num in range(max_expruns):
    try:
        x_train, x_test, y_train, y_test = read_data(run_num, prob)

        results["MLP"].append(scipy_models(x_train, x_test, y_train, y_test, 0, hidden, learn_rate, run_num, prob))
        results["DecisionTree"].append(scipy_models(x_train, x_test, y_train, y_test, 1, hidden, learn_rate, run_num, prob))
        results["RandomForest"].append(scipy_models(x_train, x_test, y_train, y_test, 2, hidden, learn_rate, run_num, prob))
        results["AdaBoost"].append(scipy_models(x_train, x_test, y_train, y_test, 3, hidden, learn_rate, run_num, prob))
        results["GBoost"].append(scipy_models(x_train, x_test, y_train, y_test, 4, hidden, learn_rate, run_num, prob))
        results["XGBoost"].append(xgboost_models(x_train, x_test, y_train, y_test, run_num, prob))

        print(f"Run {run_num} complete.")
    except Exception as e:
        print(f"Error in Run {run_num}: {e}. Ensure data files are in the /data folder.")

# Summarize Results
print("\n--- Final Results ---")
for model, scores in results.items():
    if scores:
        print(f"{model:12} | Mean: {np.mean(scores):.4f} | Std: {np.std(scores):.4f}")

Starting experiments for: classifification

Error in Run 0: name 'xgb' is not defined. Ensure data files are in the /data folder.
Error in Run 1: name 'xgb' is not defined. Ensure data files are in the /data folder.
Error in Run 2: name 'xgb' is not defined. Ensure data files are in the /data folder.
Error in Run 3: name 'xgb' is not defined. Ensure data files are in the /data folder.
Error in Run 4: name 'xgb' is not defined. Ensure data files are in the /data folder.

--- Final Results ---
MLP          | Mean: 0.6474 | Std: 0.0277
DecisionTree | Mean: 0.7325 | Std: 0.0296
RandomForest | Mean: 0.7474 | Std: 0.0234
AdaBoost     | Mean: 0.7597 | Std: 0.0141
GBoost       | Mean: 0.7539 | Std: 0.0271
